In [5]:
import os
import pandas as pd

# Dynamic path resolution: checks root path first, then parent directory
if os.path.exists(os.path.join("data", "raw", "smartphones_raw.csv")):
    RAW_DATA_PATH = os.path.join("data", "raw", "smartphones_raw.csv")
    PROCESSED_DATA_PATH = os.path.join("data", "processed", "processed_data.csv")
else:
    RAW_DATA_PATH = os.path.join("..", "data", "raw", "smartphones_raw.csv")
    PROCESSED_DATA_PATH = os.path.join("..", "data", "processed", "processed_data.csv")

# Load dataset
df_raw = pd.read_csv(RAW_DATA_PATH)
print(f"Raw dataset shape: {df_raw.shape}")
df_raw.head()

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Display dataset column information
df_raw.info()

# Check total missing values per column
print("\nMissing values per column:")
print(df_raw.isnull().sum())

In [ ]:
import re

df = df_raw.copy()

# Standardize column names (lowercase with underscores)
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Dynamically identify text and rating columns
text_col = next((col for col in ["body", "review", "review_text", "text"] if col in df.columns), None)
rating_col = next((col for col in ["rating", "ratings", "score"] if col in df.columns), None)

print(f"Using text column: '{text_col}' | rating column: '{rating_col}'")

# 1. Drop rows missing essential review text or ratings
df = df.dropna(subset=[text_col, rating_col])

# 2. Extract/Standardize Brand name
if "brand" not in df.columns and "title" in df.columns:
    df["brand"] = df["title"].astype(str).str.split().str[0].str.upper()
elif "brand" in df.columns:
    df["brand"] = df["brand"].astype(str).str.strip().str.upper()

# 3. Clean review text (remove excess spaces)
df["clean_review"] = df[text_col].astype(str).apply(lambda s: re.sub(r"\s+", " ", s).strip())

# 4. Remove duplicate reviews
df = df.drop_duplicates(subset=["clean_review"])

# 5. Add review word count feature & remove empty/short reviews
df["review_length"] = df["clean_review"].apply(lambda s: len(s.split()))
df = df[df["review_length"] >= 3]

print(f"Cleaned dataset shape: {df.shape}")
df[["brand", rating_col, "review_length", "clean_review"]].head()

In [ ]:
# Ensure output directory exists
os.makedirs(os.path.dirname(PROCESSED_DATA_PATH), exist_ok=True)

# Export cleaned data to processed folder
df.to_csv(PROCESSED_DATA_PATH, index=False)
print(f"Successfully saved clean dataset to: {PROCESSED_DATA_PATH}")